# AI Essay Detector - Unified 5-Fold Cross-Validation Pipeline

This master notebook compiles all steps of the AI essay detection pipeline. It is optimized for memory safety on Windows systems and high execution speed.

## 📖 Project Workflow & System Architecture
The workflow consists of the following modular steps:
1. **Kaggle Setup & Dataset Check**: Automatically checks and downloads the raw dataset.
2. **Step 1: Data Cleaning**: Whitespace normalization and filtering out truncated essays (< 100 words).
3. **Step 2: Stratified 5-Fold Preprocessing**: Preserves label distribution and extracts 50,000 hybrid word/character TF-IDF features with zero data leakage.
4. **Step 3: Evaluation Utilities**: Shared reporting and threshold tuning functions (enforcing a strict FPR limit $\le 0.1\%$).
5. **Step 4: Model Training**: Folds training for Logistic Regression, calibrated LinearSVC (SVM), calibrated Multinomial Naive Bayes, and Random Forest.
6. **Step 5: Model Comparison**: Performance synthesis and comparison visualizations.

### Setup Cell: Dataset Presence Check & Kaggle Downloader
Automatically downloads the DAIGT V4 raw dataset from Kaggle if it is not present in the root directory.

In [ ]:
import os

DATASET_FILE_NAME = "train_v4_drcat_01.csv"
KAGGLE_ZIP_FILE_NAME = "daigt-v4-train-dataset.zip"

# Check if the dataset file already exists locally
if os.path.exists(DATASET_FILE_NAME):
    print(f"'{DATASET_FILE_NAME}' found locally. Skipping Kaggle download.")
else:
    print(f"'{DATASET_FILE_NAME}' not found locally. Attempting to download from Kaggle...")

    # 1. Install the Kaggle library
    !pip install kaggle --quiet

    # Ensure the .kaggle directory exists (redundant if previous cell ran, but safe)
    !mkdir -p ~/.kaggle

    # The kaggle.json file should have been created by the previous cell
    # To prevent 'kaggle.json' from being publicly readable
    !chmod 600 ~/.kaggle/kaggle.json

    print("Kaggle API setup should be complete (kaggle.json created by previous cell).")

    # 3. Download the dataset
    dataset_name = "thedrcat/daigt-v4-train-dataset"
    !kaggle datasets download -d {dataset_name}

    # 4. Unzip the downloaded file
    !unzip -o {KAGGLE_ZIP_FILE_NAME} -d .

    # 5. Clean up the zip file (optional)
    !rm {KAGGLE_ZIP_FILE_NAME}

    if os.path.exists(DATASET_FILE_NAME):
        print(f"\nDataset '{dataset_name}' downloaded and unzipped successfully.")
        print(f"You should now see '{DATASET_FILE_NAME}' in your current directory.")
    else:
        raise FileNotFoundError(f"Failed to download and extract '{DATASET_FILE_NAME}' from Kaggle.")


### Library Imports & Environment Setup
We import standard computational and visualization packages, alongside custom models and scikit-learn modules.

In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, save_npz, load_npz
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configure plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

print("Setup complete. All packages imported successfully.")


## Step 1: Data Cleaning
Loads the raw essays, normalizes whitespaces, and removes corrupted inputs (fewer than 100 words).

In [ ]:
def clean_dataset(input_path, output_path):
    print("=== Loading Raw Dataset ===")
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found at {input_path}")
        
    df = pd.read_csv(input_path)
    initial_shape = df.shape
    print(f"Loaded {initial_shape[0]} rows and {initial_shape[1]} columns.")
    
    print("\n=== Preprocessing & Data Cleaning ===")
    
    # 1. Strip leading and trailing whitespace from text
    print("Normalizing whitespaces...")
    df['text'] = df['text'].astype(str).str.strip()
    
    # 2. Calculate word count for filtering
    print("Calculating word counts...")
    df['word_count'] = df['text'].apply(lambda x: len(x.split()))
    
    # 3. Identify and filter corrupted/ultra-short texts
    # We filter out text with fewer than 100 words as these are prompt titles, truncated lines, or gibberish.
    min_word_limit = 100
    corrupted_mask = df['word_count'] < min_word_limit
    num_corrupted = corrupted_mask.sum()
    
    print(f"Found {num_corrupted} rows with word count < {min_word_limit} (corrupted/truncated essays).")
    
    # Let's log some examples of filtered texts for transparency
    if num_corrupted > 0:
        print("\nExamples of filtered short/corrupted texts:")
        sample_corrupted = df[corrupted_mask].sort_values(by='word_count').head(5)
        for idx, row in sample_corrupted.iterrows():
            print(f" - [Label: {row['label']}, Source: {row['source']}, Words: {row['word_count']}] Text: {repr(row['text'][:150])}...")
            
    # Remove corrupted rows
    df_clean = df[~corrupted_mask].copy()
    
    # Drop the temporary word_count column if we want to match the original structure, 
    # but keeping it is actually very useful for exploratory analysis! 
    # Let's keep it for their model training and analysis.
    
    final_shape = df_clean.shape
    rows_removed = initial_shape[0] - final_shape[0]
    
    print(f"\nFiltered out {rows_removed} rows. Cleaned dataset has {final_shape[0]} rows.")
    
    # 4. Check label distribution in the cleaned dataset
    print("\n=== Cleaned Label Distribution ===")
    label_counts = df_clean['label'].value_counts()
    for label, count in label_counts.items():
        percentage = (count / final_shape[0]) * 100
        label_name = "AI-Generated (1)" if label == 1 else "Human-Written (0)"
        print(f" - {label_name}: {count} ({percentage:.2f}%)")
        
    # 5. Save the cleaned dataset
    print(f"\n=== Saving Cleaned Dataset ===")
    df_clean.to_csv(output_path, index=False)
    print(f"Cleaned dataset successfully saved to: {output_path}")
    print(f"File size: {os.path.getsize(output_path) / (1024 * 1024):.2f} MB")

INPUT_FILE = "train_v4_drcat_01.csv"
OUTPUT_FILE = "train_v4_drcat_01_cleaned.csv"
clean_dataset(INPUT_FILE, OUTPUT_FILE)


## Step 2: Stratified 5-Fold Preprocessing & Feature Extraction
Splits the dataset using a Stratified K-Fold setup and fits TfidfVectorizers. Fits are executed **strictly** on the training subset of each fold to prevent data leakage.

In [ ]:
def run_preprocessing(input_csv, output_dir, models_dir):
    print("=== Loading Cleaned Dataset ===")
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"Cleaned dataset not found at {input_csv}. Please run clean_dataset.py first.")
        
    df = pd.read_csv(input_csv)
    print(f"Loaded {df.shape[0]} rows.")
    
    # Fill any NaNs just in case
    df['text'] = df['text'].fillna('')
    
    print("\n=== Performing 5-Fold Stratified Split ===")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df['label'])):
        print(f"\n--- Processing Fold {fold} ---")
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        
        print(f"Training set size: {train_df.shape[0]} rows")
        print(f"Testing set size: {test_df.shape[0]} rows")
        
        # Track label distributions
        for dataset_name, subset in [("Train", train_df), ("Test", test_df)]:
            counts = subset['label'].value_counts()
            print(f"  {dataset_name} set - AI (1): {counts.get(1, 0)} ({counts.get(1, 0)/len(subset)*100:.2f}%), "
                  f"Human (0): {counts.get(0, 0)} ({counts.get(0, 0)/len(subset)*100:.2f}%)")
                  
        print("Fitting Word-level TF-IDF (n-grams: 1 to 2, max features: 25,000, stop words: english)...")
        word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
            stop_words='english'
        )
        
        # Fit only on the training set to prevent data leakage!
        X_train_word = word_vectorizer.fit_transform(train_df['text'])
        X_test_word = word_vectorizer.transform(test_df['text'])
        
        print("Fitting Character-level TF-IDF (char n-grams: 3 to 5, max features: 25,000)...")
        char_vectorizer = TfidfVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            max_features=25000,
            sublinear_tf=True
        )
        
        X_train_char = char_vectorizer.fit_transform(train_df['text'])
        X_test_char = char_vectorizer.transform(test_df['text'])
        
        print(f"Word vocabulary size: {len(word_vectorizer.vocabulary_)}")
        print(f"Char vocabulary size: {len(char_vectorizer.vocabulary_)}")
        
        print("Concatenating Word and Character Feature Matrices...")
        X_train = hstack([X_train_word, X_train_char]).tocsr()
        X_test = hstack([X_test_word, X_test_char]).tocsr()
        
        y_train = train_df['label'].values
        y_test = test_df['label'].values
        
        print(f"Combined Training Matrix Shape: {X_train.shape}")
        print(f"Combined Testing Matrix Shape: {X_test.shape}")
        
        # Setup directories for this fold
        fold_output_dir = os.path.join(output_dir, f"fold_{fold}")
        fold_models_dir = os.path.join(models_dir, f"fold_{fold}")
        os.makedirs(fold_output_dir, exist_ok=True)
        os.makedirs(fold_models_dir, exist_ok=True)
        
        # Save sparse feature matrices
        save_npz(os.path.join(fold_output_dir, "X_train.npz"), X_train)
        save_npz(os.path.join(fold_output_dir, "X_test.npz"), X_test)
        
        # Save target label arrays
        np.save(os.path.join(fold_output_dir, "y_train.npy"), y_train)
        np.save(os.path.join(fold_output_dir, "y_test.npy"), y_test)
        
        # Save indices of train/test for traceability
        np.save(os.path.join(fold_output_dir, "train_idx.npy"), train_idx)
        np.save(os.path.join(fold_output_dir, "test_idx.npy"), test_idx)
        
        # Save the fitted vectorizers using joblib
        joblib.dump(word_vectorizer, os.path.join(fold_models_dir, "word_vectorizer.joblib"))
        joblib.dump(char_vectorizer, os.path.join(fold_models_dir, "char_vectorizer.joblib"))
        
        print(f"Fold {fold} preprocessing completed successfully!")

INPUT_CSV = "train_v4_drcat_01_cleaned.csv"
OUTPUT_DIR = "processed_data"
MODELS_DIR = "models"
run_preprocessing(INPUT_CSV, OUTPUT_DIR, MODELS_DIR)


## Step 3: Shared Evaluation Utilities
This block defines metrics extraction (`calculate_metrics`), terminal reporting (`print_evaluation_report`), false positive rate threshold tuning (`tune_decision_threshold`), and global out-of-fold plot generation (`plot_global_evaluation`).

In [ ]:
def calculate_metrics(y_true, y_pred, y_prob=None):
    """
    Computes standard evaluation metrics alongside critical metrics for academic integrity
    (False Positive Rate and Specificity) to minimize false accusations.
    """
    # Confusion Matrix: TN, FP, FN, TP
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Core calculations
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    # Specificity = TN / (TN + FP) -> proportion of true human essays correctly classified
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # False Positive Rate = FP / (FP + TN) -> proportion of human essays falsely accused
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    # Precision = TP / (TP + FP) -> proportion of flagged essays that are actually AI
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    # Recall (Sensitivity) = TP / (TP + FN) -> proportion of AI essays detected
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else None
    
    metrics = {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'false_positive_rate': fpr,
        'auc_roc': auc,
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    }
    return metrics

def print_evaluation_report(model_name, metrics):
    """
    Prints a detailed, highly readable terminal report of a model's performance.
    """
    print("=" * 60)
    print(f" EVALUATION REPORT FOR: {model_name.upper()} ")
    print("=" * 60)
    print(f"Accuracy:                 {metrics['accuracy']*100:.3f}%")
    print(f"F1-Score:                 {metrics['f1_score']:.4f}")
    if metrics['auc_roc'] is not None:
        print(f"ROC-AUC:                  {metrics['auc_roc']:.4f}")
    
    print("\n--- Stylistic & Academic Integrity Safety Metrics ---")
    print(f"Precision (Positive Predictive Value):  {metrics['precision']*100:.3f}%")
    print(f"Recall / Sensitivity (Detection Rate):  {metrics['recall']*100:.3f}%")
    print(f"Specificity (True Negative Rate):       {metrics['specificity']*100:.3f}%")
    
    # Highlight False Positive Rate (False Accusations)
    fpr_pct = metrics['false_positive_rate'] * 100
    print("-" * 60)
    if fpr_pct > 1.0:
        print(f"[!] False Positive Rate (Falsely Accused): {fpr_pct:.3f}%  (HIGH RISK!)")
    elif fpr_pct > 0.1:
        print(f"[!] False Positive Rate (Falsely Accused): {fpr_pct:.3f}%  (MODERATE RISK)")
    else:
        print(f"[OK] False Positive Rate (Falsely Accused): {fpr_pct:.3f}%  (SAFE / LOW RISK)")
    print("-" * 60)
    
    cm = metrics['confusion_matrix']
    print(f"\nConfusion Matrix:")
    print(f"  Predicted Human  |  TN: {cm['tn']}  |  FN: {cm['fn']}")
    print(f"  Predicted AI     |  FP: {cm['fp']}  |  TP: {cm['tp']}")
    print("=" * 60)

def tune_decision_threshold(y_true, y_prob, target_fpr=0.001):
    """
    Scans different probability thresholds to find the threshold that guarantees 
    the False Positive Rate (FPR) is at or below the 'target_fpr'.
    Locks down the threshold to protect innocent students.
    """
    print(f"\nScanning thresholds to enforce False Positive Rate <= {target_fpr*100:.3f}%...")
    
    # Sort probabilities
    thresholds = np.linspace(0.0, 1.0, 1001)
    best_threshold = 0.5
    best_metrics = None
    found = False
    
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        metrics = calculate_metrics(y_true, y_pred, y_prob)
        
        # We want the threshold that yields FPR <= target_fpr while maximizing recall
        if metrics['false_positive_rate'] <= target_fpr:
            if not found or metrics['recall'] > best_metrics['recall']:
                best_threshold = t
                best_metrics = metrics
                found = True
                
    if found:
        print(f"[OK] Found Optimal Academic Integrity Threshold: {best_threshold:.3f}")
        print(f"   Recall (Detection Rate) at this threshold: {best_metrics['recall']*100:.2f}%")
        print(f"   Actual False Positive Rate achieved:       {best_metrics['false_positive_rate']*100:.3f}%")
        return best_threshold, best_metrics
    else:
        print("[!] Could not satisfy the exact FPR target with current model predictions. Using fallback threshold of 0.99.")
        fallback_pred = (y_prob >= 0.99).astype(int)
        return 0.99, calculate_metrics(y_true, fallback_pred, y_prob)


def plot_global_evaluation(y_true, y_prob, threshold, model_name, filename_prefix):
    """
    Plots the global Out-of-Fold ROC Curve (with AUC) and Confusion Matrix (tuned threshold)
    and saves them to the results/ folder.
    """
    import os
    from sklearn.metrics import roc_curve, confusion_matrix, auc
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Create results dir if not exists
    os.makedirs("results", exist_ok=True)
    
    # 1. Confusion Matrix
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
    plt.title(f"{model_name} Confusion Matrix\n(Tuned Threshold = {threshold:.3f})")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.tight_layout()
    cm_path = f"results/{filename_prefix}_confusion_matrix.png"
    plt.savefig(cm_path, dpi=150)
    plt.close()
    
    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} ROC Curve (Global OOF)')
    plt.legend(loc="lower right")
    plt.tight_layout()
    roc_path = f"results/{filename_prefix}_roc_curve.png"
    plt.savefig(roc_path, dpi=150)
    plt.close()
    
    print(f"Saved evaluation plots for {model_name} to results/")


## Step 4: Model Training and Out-of-Fold Evaluation
### Model 1: Logistic Regression Pipeline
We run a 5-Fold training process with L2 penalty, utilizing a nested 3-Fold Grid Search to tune regularization strength $C$.

In [ ]:
def train_logistic_regression(data_dir, models_dir):
    print("=== Member 1: Loading Shared Preprocessed Folds ===")
    
    n_folds = 5
    oof_probs = []
    oof_labels = []
    fold_metrics_list = []
    
    print(f"Starting 5-Fold Cross-Validation training...")
    
    for fold in range(n_folds):
        print(f"\n========================================")
        print(f" TRAINING FOLD {fold}")
        print(f"========================================")
        fold_data_dir = os.path.join(data_dir, f"fold_{fold}")
        fold_models_dir = os.path.join(models_dir, f"fold_{fold}")
        
        X_train_path = os.path.join(fold_data_dir, "X_train.npz")
        X_test_path = os.path.join(fold_data_dir, "X_test.npz")
        y_train_path = os.path.join(fold_data_dir, "y_train.npy")
        y_test_path = os.path.join(fold_data_dir, "y_test.npy")
        
        if not (os.path.exists(X_train_path) and os.path.exists(X_test_path)):
            raise FileNotFoundError(f"Preprocessed sparse matrices for Fold {fold} not found. Please run preprocess.py first.")
            
        X_train = load_npz(X_train_path)
        X_test = load_npz(X_test_path)
        y_train = np.load(y_train_path)
        y_test = np.load(y_test_path)
        
        print(f"X_train Shape: {X_train.shape}")
        print(f"X_test Shape:  {X_test.shape}")
        print(f"y_train distribution: AI (1)={np.sum(y_train)}, Human (0)={len(y_train) - np.sum(y_train)}")
        
        # Define hyperparameter grid for Logistic Regression
        param_grid = {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2']
        }
        
        base_model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
        
        # 3-Fold CV on the training fold (nested CV)
        print(f"Running GridSearchCV for Fold {fold} with 3-fold cross-validation...")
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            scoring='roc_auc',
            cv=3,
            n_jobs=1,
            verbose=0
        )
        
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        
        print(f"Best Hyperparameters for Fold {fold}: {grid_search.best_params_}")
        print(f"Best CV ROC-AUC for Fold {fold}:            {grid_search.best_score_:.4f}")
        
        # Predict default probabilities and classes
        y_prob = best_model.predict_proba(X_test)[:, 1]
        y_pred_default = (y_prob >= 0.5).astype(int)
        
        oof_probs.append(y_prob)
        oof_labels.append(y_test)
        
        # Calculate individual fold metrics (at default threshold 0.5)
        fold_metrics = calculate_metrics(y_test, y_pred_default, y_prob)
        fold_metrics_list.append(fold_metrics)
        
        # Save model for this fold
        os.makedirs(fold_models_dir, exist_ok=True)
        model_save_path = os.path.join(fold_models_dir, "logistic_regression_model.joblib")
        joblib.dump(best_model, model_save_path)
        print(f"Fold {fold} Model saved successfully to: {model_save_path}")
        
    # Concatenate Out-Of-Fold (OOF) arrays
    oof_y_prob = np.concatenate(oof_probs)
    oof_y_true = np.concatenate(oof_labels)
    
    print("\n========================================")
    print(" FOLD-BY-FOLD SUMMARY (Default Threshold = 0.5)")
    print("========================================")
    for fold, fm in enumerate(fold_metrics_list):
        print(f"Fold {fold} | Accuracy: {fm['accuracy']*100:.2f}% | F1: {fm['f1_score']:.4f} | ROC-AUC: {fm['auc_roc']:.4f} | FPR: {fm['false_positive_rate']*100:.3f}%")
        
    print("\n========================================")
    print(" PHASE 2: GLOBAL OUT-OF-FOLD EVALUATION (Default Threshold = 0.5)")
    print("========================================")
    oof_default_metrics = calculate_metrics(oof_y_true, (oof_y_prob >= 0.5).astype(int), oof_y_prob)
    print_evaluation_report("Logistic Regression (Global OOF - Default)", oof_default_metrics)
    
    print("\n========================================")
    print(" PHASE 3: GLOBAL ACADEMIC INTEGRITY TUNING (FPR <= 0.1%)")
    print("========================================")
    target_fpr = 0.001
    best_threshold, tuned_metrics = tune_decision_threshold(oof_y_true, oof_y_prob, target_fpr=target_fpr)
    print_evaluation_report(f"Logistic Regression (Global OOF - Tuned Threshold @ {best_threshold:.3f})", tuned_metrics)
    
    # Generate and save comparative plots (confusion matrix & ROC curve)
    from evaluate import plot_global_evaluation
    plot_global_evaluation(oof_y_true, oof_y_prob, best_threshold, "Logistic Regression", "logistic")
    
    # Save results to json summary matching the format of other models
    OUTPUT_DIR = "results"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    summary_path = os.path.join(OUTPUT_DIR, "logistic_cross_fold_summary.json")
    
    metrics_keys = ["accuracy", "f1_score", "auc_roc", "precision",
                    "recall", "specificity", "false_positive_rate"]
    
    summary_export = []
    for fold, fm in enumerate(fold_metrics_list):
        y_prob_fold = oof_probs[fold]
        y_test_fold = oof_labels[fold]
        tuned_fm = calculate_metrics(y_test_fold, (y_prob_fold >= best_threshold).astype(int), y_prob_fold)
        summary_export.append({
            "fold": fold,
            "best_threshold": best_threshold,
            "cv_auc": fm["auc_roc"],
            "metrics_default": {k: fm[k] for k in metrics_keys},
            "metrics_tuned": {k: tuned_fm[k] for k in metrics_keys},
        })
        
    global_results = {
        "model_name": "Logistic Regression",
        "global_default": {k: oof_default_metrics[k] for k in metrics_keys},
        "global_tuned": {k: tuned_metrics[k] for k in metrics_keys},
        "global_tuned_threshold": best_threshold,
        "fold_results": summary_export
    }
    
    import json
    with open(summary_path, "w") as f:
        json.dump(global_results, f, indent=4)
    print(f"\nCross-fold summary saved -> {summary_path}")

DATA_DIR = "processed_data"
MODELS_DIR = "models"
lr_true, lr_prob, lr_metrics = train_logistic_regression(DATA_DIR, MODELS_DIR)


### Model 2: Support Vector Machine (LinearSVC)
Trains a Linear SVM. Hyperparameters are tuned on the classifier directly, and calibration (Platt scaling) is run once per fold (`n_jobs=1` is utilized for Windows memory safety).

In [ ]:
def tune_hyperparameters(X_train, y_train, fold: int):
    """
    Run GridSearchCV over the LinearSVC param grid.
    LinearSVC does not natively support predict_proba, but GridSearchCV can use
    decision_function for 'roc_auc' scoring.
    Optimises for ROC-AUC (robust to class imbalance).
    Returns the best estimator and its parameters.
    """
    n_combinations = len(PARAM_GRID["C"])
    print(f"\n  [Fold {fold}] Starting GridSearchCV over {n_combinations} "
          f"hyperparameter combinations (3-fold inner CV)...")

    base_model = LinearSVC(max_iter=3000, random_state=42)

    # Inner 3-fold CV for hyperparameter search (on training data only — no leakage)
    grid_search = GridSearchCV(
        estimator  = base_model,
        param_grid = PARAM_GRID,
        scoring    = "roc_auc",
        cv         = 3,
        n_jobs     = 1,
        verbose    = 0,
        refit      = True,
    )
    grid_search.fit(X_train, y_train)

    best_params = grid_search.best_params_
    best_score  = grid_search.best_score_

    print(f"  [Fold {fold}] Best params : {best_params}")
    print(f"  [Fold {fold}] Best CV AUC : {best_score:.4f}")

    return grid_search.best_estimator_, best_params, best_score


def calibrate_model(best_estimator, X_train, y_train):
    """
    Wrap the tuned LinearSVC in Platt scaling (sigmoid calibration) so that
    predict_proba() outputs are well-calibrated probabilities.
    This is critical for the FPR threshold tuning step.
    """
    print("  Calibrating probability outputs (Platt scaling / sigmoid)...")
    from sklearn.frozen import FrozenEstimator
    frozen_estimator = FrozenEstimator(best_estimator)
    
    # Create a custom cv fold that trains/calibrates on the full training set
    n_samples = X_train.shape[0]
    indices = np.arange(n_samples)
    custom_cv = [(indices, indices)]
    
    calibrated = CalibratedClassifierCV(
        estimator = frozen_estimator,
        method    = "sigmoid",
        cv        = custom_cv,
    )
    calibrated.fit(X_train, y_train)
    return calibrated


def save_fold_results(fold: int, model, best_params: dict, metrics_default: dict,
                      metrics_tuned: dict, best_threshold: float):
    """Persist the trained model and metrics for this fold."""
    fold_model_dir = os.path.join(MODELS_DIR, f"fold_{fold}")
    os.makedirs(fold_model_dir, exist_ok=True)

    # Save calibrated SVM model
    model_path = os.path.join(fold_model_dir, "svm_model.joblib")
    joblib.dump(model, model_path)

    # Save metrics + config as JSON
    fold_results_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    os.makedirs(fold_results_dir, exist_ok=True)

    result_record = {
        "fold"            : fold,
        "best_params"     : best_params,
        "best_threshold"  : best_threshold,
        "metrics_default" : {k: v for k, v in metrics_default.items() if k != "confusion_matrix"},
        "metrics_tuned"   : {k: v for k, v in metrics_tuned.items()   if k != "confusion_matrix"},
        "confusion_matrix_default": metrics_default["confusion_matrix"],
        "confusion_matrix_tuned"  : metrics_tuned["confusion_matrix"],
    }

    json_path = os.path.join(fold_results_dir, "svm_results.json")
    with open(json_path, "w") as f:
        json.dump(result_record, f, indent=4)

    print(f"  Model saved   -> {model_path}")
    print(f"  Metrics saved -> {json_path}")



# =============================================================================
# MAIN TRAINING LOOP

# =============================================================================

def run_training():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 65)
    print(f"  MODEL: {MODEL_NAME}")
    print(f"  Folds : {N_FOLDS}  |  Target FPR ceiling: {TARGET_FPR*100:.1f}%")
    print("=" * 65)

    # Accumulators for cross-fold summary
    fold_summaries = []
    
    # Global OOF prediction accumulators
    oof_probs = []
    oof_labels = []

    for fold in range(N_FOLDS):
        print(f"\n{'='*65}")
        print(f"  FOLD {fold} / {N_FOLDS - 1}")
        print(f"{'='*65}")

        # ------------------------------------------------------------------
        # 1. Load preprocessed data
        # ------------------------------------------------------------------
        print(f"\n[1/5] Loading fold {fold} data...")
        X_train, X_test, y_train, y_test = load_fold_data(fold)
        print(f"  X_train: {X_train.shape}  |  X_test: {X_test.shape}")
        print(f"  y_train - AI: {y_train.sum()}  Human: {(y_train==0).sum()}")
        print(f"  y_test  - AI: {y_test.sum()}   Human: {(y_test==0).sum()}")

        # ------------------------------------------------------------------
        # 2. Hyperparameter tuning (inner CV on training fold only)
        # ------------------------------------------------------------------
        print(f"\n[2/5] Hyperparameter tuning...")
        best_estimator, best_params, best_cv_auc = tune_hyperparameters(X_train, y_train, fold)

        # ------------------------------------------------------------------
        # 3. Probability calibration
        # ------------------------------------------------------------------
        print(f"\n[3/5] Probability calibration...")
        calibrated_model = calibrate_model(best_estimator, X_train, y_train)

        # ------------------------------------------------------------------
        # 4. Evaluate with default threshold (0.5)
        # ------------------------------------------------------------------
        print(f"\n[4/5] Evaluating at default threshold (0.5)...")
        y_prob_test    = calibrated_model.predict_proba(X_test)[:, 1]
        y_pred_default = (y_prob_test >= 0.5).astype(int)
        
        # Accumulate fold predictions
        oof_probs.append(y_prob_test)
        oof_labels.append(y_test)
        
        metrics_default = calculate_metrics(y_test, y_pred_default, y_prob_test)
        print_evaluation_report(f"{MODEL_NAME} - Fold {fold} (threshold=0.50)", metrics_default)

        # ------------------------------------------------------------------
        # 5. Tune decision threshold to enforce FPR ceiling
        # ------------------------------------------------------------------
        print(f"\n[5/5] Tuning decision threshold (target FPR <= {TARGET_FPR*100:.1f}%)...")
        best_threshold, metrics_tuned = tune_decision_threshold(y_test, y_prob_test, target_fpr=TARGET_FPR)
        print_evaluation_report(
            f"{MODEL_NAME} - Fold {fold} (threshold={best_threshold:.3f}, FPR-tuned)",
            metrics_tuned
        )

        # ------------------------------------------------------------------
        # 6. Persist model + metrics
        # ------------------------------------------------------------------
        print(f"\n[6/5] Saving model and results...")
        save_fold_results(fold, calibrated_model, best_params,
                          metrics_default, metrics_tuned, best_threshold)

        fold_summaries.append({
            "fold"           : fold,
            "best_params"    : best_params,
            "best_threshold" : best_threshold,
            "cv_auc"         : best_cv_auc,
            "default"        : metrics_default,
            "tuned"          : metrics_tuned,
        })

    # ==========================================================================
    # GLOBAL OUT-OF-FOLD EVALUATION & TUNING
    # ==========================================================================
    oof_y_prob = np.concatenate(oof_probs)
    oof_y_true = np.concatenate(oof_labels)
    
    print("\n" + "=" * 65)
    print("  GLOBAL OUT-OF-FOLD EVALUATION (Default Threshold = 0.5)")
    print("=" * 65)
    global_default_metrics = calculate_metrics(oof_y_true, (oof_y_prob >= 0.5).astype(int), oof_y_prob)
    print_evaluation_report(f"{MODEL_NAME} (Global OOF - Default)", global_default_metrics)
    
    print("\n" + "=" * 65)
    print(f"  GLOBAL ACADEMIC INTEGRITY TUNING (FPR <= {TARGET_FPR*100:.1f}%)")
    print("=" * 65)
    global_best_threshold, global_tuned_metrics = tune_decision_threshold(oof_y_true, oof_y_prob, target_fpr=TARGET_FPR)
    print_evaluation_report(f"{MODEL_NAME} (Global OOF - Tuned @ {global_best_threshold:.3f})", global_tuned_metrics)
    
    # Generate and save plots (confusion matrix & ROC curve)
    from evaluate import plot_global_evaluation
    plot_global_evaluation(oof_y_true, oof_y_prob, global_best_threshold, MODEL_NAME, "svm")

    # ==========================================================================
    # CROSS-FOLD SUMMARY
    # ==========================================================================
    print("\n" + "=" * 65)
    print(f"  CROSS-FOLD SUMMARY - {MODEL_NAME}")
    print("=" * 65)

    metrics_keys = ["accuracy", "f1_score", "auc_roc", "precision",
                    "recall", "specificity", "false_positive_rate"]

    print(f"\n{'Metric':<30} {'Mean (default)':>15} {'Mean (tuned)':>14}")
    print("-" * 62)
    for key in metrics_keys:
        default_vals = [s["default"][key] for s in fold_summaries if s["default"][key] is not None]
        tuned_vals   = [s["tuned"][key]   for s in fold_summaries if s["tuned"][key]   is not None]
        mean_def = np.mean(default_vals) if default_vals else float("nan")
        mean_tun = np.mean(tuned_vals)   if tuned_vals   else float("nan")
        label = key.replace("_", " ").title()
        print(f"  {label:<28} {mean_def:>13.4f}  {mean_tun:>13.4f}")

    # Best hyperparameters across folds
    print("\nBest Hyperparameters per Fold:")
    for s in fold_summaries:
        print(f"  Fold {s['fold']}: C={s['best_params']['C']}, "
              f"threshold={s['best_threshold']:.3f}, "
              f"CV-AUC={s['cv_auc']:.4f}")

    # Save cross-fold summary
    summary_path = os.path.join(OUTPUT_DIR, "svm_cross_fold_summary.json")
    summary_export = []
    for s in fold_summaries:
        summary_export.append({
            "fold"           : s["fold"],
            "best_params"    : s["best_params"],
            "best_threshold" : s["best_threshold"],
            "cv_auc"         : s["cv_auc"],
            "metrics_default": {k: s["default"][k] for k in metrics_keys},
            "metrics_tuned"  : {k: s["tuned"][k]   for k in metrics_keys},
        })
        
    global_results = {
        "model_name": MODEL_NAME,
        "global_default": {k: global_default_metrics[k] for k in metrics_keys},
        "global_tuned": {k: global_tuned_metrics[k] for k in metrics_keys},
        "global_tuned_threshold": global_best_threshold,
        "fold_results": summary_export
    }
    
    with open(summary_path, "w") as f:
        json.dump(global_results, f, indent=4)
    print(f"\nCross-fold summary saved -> {summary_path}")
    print("=" * 65)
    print("  Training complete!")
    print("=" * 65)



# =============================================================================
# ENTRY POINT

# =============================================================================

svm_true, svm_prob, svm_metrics = run_training()


### Model 3: Multinomial Naive Bayes
Fits a Naive Bayes model. Tuning smoothing constant $\alpha$ with nested CV (`n_jobs=1` to prevent crashes). Explains probability bounds under safety thresholds.

In [ ]:
def tune_hyperparameters(X_train, y_train, fold: int):
    """
    Run GridSearchCV over the Multinomial NB param grid.
    Optimises for ROC-AUC (robust to class imbalance).
    Returns the best estimator and its parameters.
    """
    print(f"\n  [Fold {fold}] Starting GridSearchCV over {len(PARAM_GRID['alpha']) * len(PARAM_GRID['fit_prior'])} "
          f"hyperparameter combinations (3-fold inner CV)...")
    
    base_model = MultinomialNB()
    
    # Inner 3-fold CV for hyperparameter search (on training data only — no leakage)
    grid_search = GridSearchCV(
        estimator  = base_model,
        param_grid = PARAM_GRID,
        scoring    = "roc_auc",
        cv         = 3,
        n_jobs     = 1,
        verbose    = 0,
        refit      = True,
    )
    grid_search.fit(X_train, y_train)
    
    best_params = grid_search.best_params_
    best_score  = grid_search.best_score_
    
    print(f"  [Fold {fold}] Best params : {best_params}")
    print(f"  [Fold {fold}] Best CV AUC : {best_score:.4f}")
    
    return grid_search.best_estimator_, best_params, best_score


def calibrate_model(best_estimator, X_train, y_train):
    """
    Wrap the tuned MNB in Platt scaling (sigmoid calibration) so that
    predict_proba() outputs are well-calibrated probabilities.
    This is critical for the FPR threshold tuning step.
    """
    print("  Calibrating probability outputs (Platt scaling / sigmoid)...")
    from sklearn.frozen import FrozenEstimator
    frozen_estimator = FrozenEstimator(best_estimator)
    
    # Create a custom cv fold that trains/calibrates on the full training set
    n_samples = X_train.shape[0]
    indices = np.arange(n_samples)
    custom_cv = [(indices, indices)]
    
    calibrated = CalibratedClassifierCV(
        estimator = frozen_estimator,
        method    = "sigmoid",
        cv        = custom_cv,
    )
    calibrated.fit(X_train, y_train)
    return calibrated


def save_fold_results(fold: int, model, best_params: dict, metrics_default: dict,
                      metrics_tuned: dict, best_threshold: float):
    """Persist the trained model and metrics for this fold."""
    fold_model_dir = os.path.join(MODELS_DIR, f"fold_{fold}")
    os.makedirs(fold_model_dir, exist_ok=True)
    
    # Save calibrated model
    model_path = os.path.join(fold_model_dir, "mnb_model.joblib")
    joblib.dump(model, model_path)
    
    # Save metrics + config as JSON
    fold_results_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    os.makedirs(fold_results_dir, exist_ok=True)
    
    result_record = {
        "fold"            : fold,
        "best_params"     : best_params,
        "best_threshold"  : best_threshold,
        "metrics_default" : {k: v for k, v in metrics_default.items() if k != "confusion_matrix"},
        "metrics_tuned"   : {k: v for k, v in metrics_tuned.items()   if k != "confusion_matrix"},
        "confusion_matrix_default": metrics_default["confusion_matrix"],
        "confusion_matrix_tuned"  : metrics_tuned["confusion_matrix"],
    }
    
    json_path = os.path.join(fold_results_dir, "mnb_results.json")
    with open(json_path, "w") as f:
        json.dump(result_record, f, indent=4)
    
    print(f"  Model saved  -> {model_path}")
    print(f"  Metrics saved -> {json_path}")



# =============================================================================
# MAIN TRAINING LOOP

# =============================================================================

def run_training():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("=" * 65)
    print(f"  MODEL: {MODEL_NAME}")
    print(f"  Folds : {N_FOLDS}  |  Target FPR ceiling: {TARGET_FPR*100:.1f}%")
    print("=" * 65)
    
    # Accumulators for cross-fold summary
    fold_summaries = []
    
    # Global OOF prediction accumulators
    oof_probs = []
    oof_labels = []
    
    for fold in range(N_FOLDS):
        print(f"\n{'='*65}")
        print(f"  FOLD {fold} / {N_FOLDS - 1}")
        print(f"{'='*65}")
        
        # ------------------------------------------------------------------
        # 1. Load preprocessed data
        # ------------------------------------------------------------------
        print(f"\n[1/5] Loading fold {fold} data...")
        X_train, X_test, y_train, y_test = load_fold_data(fold)
        print(f"  X_train: {X_train.shape}  |  X_test: {X_test.shape}")
        print(f"  y_train - AI: {y_train.sum()}  Human: {(y_train==0).sum()}")
        print(f"  y_test  - AI: {y_test.sum()}   Human: {(y_test==0).sum()}")
        
        # ------------------------------------------------------------------
        # 2. Hyperparameter tuning (inner CV on training fold only)
        # ------------------------------------------------------------------
        print(f"\n[2/5] Hyperparameter tuning...")
        best_estimator, best_params, best_cv_auc = tune_hyperparameters(X_train, y_train, fold)
        
        # ------------------------------------------------------------------
        # 3. Probability calibration
        # ------------------------------------------------------------------
        print(f"\n[3/5] Probability calibration...")
        calibrated_model = calibrate_model(best_estimator, X_train, y_train)
        
        # ------------------------------------------------------------------
        # 4. Evaluate with default threshold (0.5)
        # ------------------------------------------------------------------
        print(f"\n[4/5] Evaluating at default threshold (0.5)...")
        y_prob_test  = calibrated_model.predict_proba(X_test)[:, 1]
        y_pred_default = (y_prob_test >= 0.5).astype(int)
        
        # Accumulate fold predictions
        oof_probs.append(y_prob_test)
        oof_labels.append(y_test)
        
        metrics_default = calculate_metrics(y_test, y_pred_default, y_prob_test)
        print_evaluation_report(f"{MODEL_NAME} - Fold {fold} (threshold=0.50)", metrics_default)
        
        # ------------------------------------------------------------------
        # 5. Tune decision threshold to enforce FPR ceiling
        # ------------------------------------------------------------------
        print(f"\n[5/5] Tuning decision threshold (target FPR <= {TARGET_FPR*100:.1f}%)...")
        best_threshold, metrics_tuned = tune_decision_threshold(y_test, y_prob_test, target_fpr=TARGET_FPR)
        print_evaluation_report(
            f"{MODEL_NAME} - Fold {fold} (threshold={best_threshold:.3f}, FPR-tuned)",
            metrics_tuned
        )
        
        # ------------------------------------------------------------------
        # 6. Persist model + metrics
        # ------------------------------------------------------------------
        save_fold_results(fold, calibrated_model, best_params,
                          metrics_default, metrics_tuned, best_threshold)
        
        fold_summaries.append({
            "fold"           : fold,
            "best_params"    : best_params,
            "best_threshold" : best_threshold,
            "cv_auc"         : best_cv_auc,
            "default"        : metrics_default,
            "tuned"          : metrics_tuned,
        })
    
    # ==========================================================================
    # GLOBAL OUT-OF-FOLD EVALUATION & TUNING
    # ==========================================================================
    oof_y_prob = np.concatenate(oof_probs)
    oof_y_true = np.concatenate(oof_labels)
    
    print("\n" + "=" * 65)
    print("  GLOBAL OUT-OF-FOLD EVALUATION (Default Threshold = 0.5)")
    print("=" * 65)
    global_default_metrics = calculate_metrics(oof_y_true, (oof_y_prob >= 0.5).astype(int), oof_y_prob)
    print_evaluation_report(f"{MODEL_NAME} (Global OOF - Default)", global_default_metrics)
    
    print("\n" + "=" * 65)
    print(f"  GLOBAL ACADEMIC INTEGRITY TUNING (FPR <= {TARGET_FPR*100:.1f}%)")
    print("=" * 65)
    global_best_threshold, global_tuned_metrics = tune_decision_threshold(oof_y_true, oof_y_prob, target_fpr=TARGET_FPR)
    print_evaluation_report(f"{MODEL_NAME} (Global OOF - Tuned @ {global_best_threshold:.3f})", global_tuned_metrics)
    
    # Generate and save plots (confusion matrix & ROC curve)
    from evaluate import plot_global_evaluation
    plot_global_evaluation(oof_y_true, oof_y_prob, global_best_threshold, MODEL_NAME, "mnb")

    # ==========================================================================
    # CROSS-FOLD SUMMARY
    # ==========================================================================
    print("\n" + "=" * 65)
    print(f"  CROSS-FOLD SUMMARY - {MODEL_NAME}")
    print("=" * 65)
    
    metrics_keys = ["accuracy", "f1_score", "auc_roc", "precision",
                    "recall", "specificity", "false_positive_rate"]
    
    print(f"\n{'Metric':<30} {'Mean (default)':>15} {'Mean (tuned)':>14}")
    print("-" * 62)
    for key in metrics_keys:
        default_vals = [s["default"][key] for s in fold_summaries if s["default"][key] is not None]
        tuned_vals   = [s["tuned"][key]   for s in fold_summaries if s["tuned"][key]   is not None]
        mean_def = np.mean(default_vals) if default_vals else float("nan")
        mean_tun = np.mean(tuned_vals)   if tuned_vals   else float("nan")
        label = key.replace("_", " ").title()
        print(f"  {label:<28} {mean_def:>13.4f}  {mean_tun:>13.4f}")
    
    # Best hyperparameters across folds
    print("\nBest Hyperparameters per Fold:")
    for s in fold_summaries:
        print(f"  Fold {s['fold']}: alpha={s['best_params']['alpha']}, "
              f"fit_prior={s['best_params']['fit_prior']}, "
              f"threshold={s['best_threshold']:.3f}, "
              f"CV-AUC={s['cv_auc']:.4f}")
    
    # Save cross-fold summary
    summary_path = os.path.join(OUTPUT_DIR, "mnb_cross_fold_summary.json")
    summary_export = []
    for s in fold_summaries:
        summary_export.append({
            "fold"           : s["fold"],
            "best_params"    : s["best_params"],
            "best_threshold" : s["best_threshold"],
            "cv_auc"         : s["cv_auc"],
            "metrics_default": {k: s["default"][k] for k in metrics_keys},
            "metrics_tuned"  : {k: s["tuned"][k]   for k in metrics_keys},
        })
        
    global_results = {
        "model_name": MODEL_NAME,
        "global_default": {k: global_default_metrics[k] for k in metrics_keys},
        "global_tuned": {k: global_tuned_metrics[k] for k in metrics_keys},
        "global_tuned_threshold": global_best_threshold,
        "fold_results": summary_export
    }
    
    with open(summary_path, "w") as f:
        json.dump(global_results, f, indent=4)
    print(f"\nCross-fold summary saved -> {summary_path}")
    print("=" * 65)
    print("  Training complete!")
    print("=" * 65)



# =============================================================================
# ENTRY POINT

# =============================================================================

run_training()


### Model 4: Random Forest Classifier
Ensemble random forest training over the 5 folds with constraints on tree depth (`max_depth=20`) to prevent RAM limits and high-dimensional crashes.

In [ ]:
def save_fold_plots(fold: int, y_true, y_pred, feature_importances=None, feature_names=None):
    """Generate and save evaluation plots for the fold instead of blocking GUI."""
    fold_results_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    os.makedirs(fold_results_dir, exist_ok=True)
    
    # 1. Confusion Matrix Plot
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Human', 'AI'],
        yticklabels=['Human', 'AI']
    )
    plt.title(f"Random Forest Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.tight_layout()
    cm_plot_path = os.path.join(fold_results_dir, "rf_confusion_matrix.png")
    plt.savefig(cm_plot_path, dpi=150)
    plt.close()
    
    # 2. Feature Importance Plot (if available)
    if feature_importances is not None and feature_names is not None:
        top_n = 20
        indices = np.argsort(feature_importances)[-top_n:]
        plt.figure(figsize=(10, 6))
        plt.barh(
            range(top_n),
            feature_importances[indices]
        )
        plt.yticks(
            range(top_n),
            [feature_names[i] for i in indices]
        )
        plt.xlabel("Importance Score")
        plt.ylabel("Features")
        plt.title(f"Top 20 Important Features - Random Forest - Fold {fold}")
        plt.tight_layout()
        fi_plot_path = os.path.join(fold_results_dir, "rf_feature_importance.png")
        plt.savefig(fi_plot_path, dpi=150)
        plt.close()


def save_fold_results(fold: int, model, metrics_default: dict,
                      metrics_tuned: dict, best_threshold: float):
    """Persist the trained model and metrics for this fold."""
    fold_model_dir = os.path.join(MODELS_DIR, f"fold_{fold}")
    os.makedirs(fold_model_dir, exist_ok=True)
    
    # Save RF model
    model_path = os.path.join(fold_model_dir, "random_forest_model.joblib")
    joblib.dump(model, model_path)
    
    # Save metrics as JSON
    fold_results_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    os.makedirs(fold_results_dir, exist_ok=True)
    
    result_record = {
        "fold"            : fold,
        "best_threshold"  : best_threshold,
        "metrics_default" : {k: v for k, v in metrics_default.items() if k != "confusion_matrix"},
        "metrics_tuned"   : {k: v for k, v in metrics_tuned.items()   if k != "confusion_matrix"},
        "confusion_matrix_default": metrics_default["confusion_matrix"],
        "confusion_matrix_tuned"  : metrics_tuned["confusion_matrix"],
    }
    
    json_path = os.path.join(fold_results_dir, "rf_results.json")
    with open(json_path, "w") as f:
        json.dump(result_record, f, indent=4)
    
    print(f"  Model saved   -> {model_path}")
    print(f"  Metrics saved -> {json_path}")



# =============================================================================
# MAIN TRAINING LOOP

# =============================================================================

def run_training():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("=" * 65)
    print(f"  MODEL: {MODEL_NAME}")
    print(f"  Folds : {N_FOLDS}  |  Target FPR ceiling: {TARGET_FPR*100:.1f}%")
    print("=" * 65)
    
    # Accumulators for cross-fold summary
    fold_summaries = []
    
    # Global OOF prediction accumulators
    oof_probs = []
    oof_labels = []
    
    for fold in range(N_FOLDS):
        print(f"\n{'='*65}")
        print(f"  FOLD {fold} / {N_FOLDS - 1}")
        print(f"{'='*65}")
        
        # 1. Load preprocessed data
        print(f"\n[1/5] Loading fold {fold} data...")
        X_train, X_test, y_train, y_test = load_fold_data(fold)
        print(f"  X_train: {X_train.shape}  |  X_test: {X_test.shape}")
        print(f"  y_train - AI: {y_train.sum()}  Human: {(y_train==0).sum()}")
        
        # 2. Train Random Forest model
        # Using optimized parameters for fast execution and memory safety on sparse data
        print(f"\n[2/5] Training Random Forest model...")
        model = RandomForestClassifier(
            n_estimators=150,
            max_depth=20,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        print("  Training Complete!")
        
        # 3. Predict probabilities
        print(f"\n[3/5] Evaluating fold predictions...")
        y_prob_test = model.predict_proba(X_test)[:, 1]
        y_pred_default = (y_prob_test >= 0.5).astype(int)
        
        oof_probs.append(y_prob_test)
        oof_labels.append(y_test)
        
        # Calculate fold metrics (at default threshold 0.5)
        metrics_default = calculate_metrics(y_test, y_pred_default, y_prob_test)
        print_evaluation_report(f"{MODEL_NAME} - Fold {fold} (threshold=0.50)", metrics_default)
        
        # Tune decision threshold for individual fold (optional / comparison)
        best_threshold, metrics_tuned = tune_decision_threshold(y_test, y_prob_test, target_fpr=TARGET_FPR)
        print_evaluation_report(
            f"{MODEL_NAME} - Fold {fold} (threshold={best_threshold:.3f}, FPR-tuned)",
            metrics_tuned
        )
        
        # 4. Generate and save plots
        print(f"\n[4/5] Generating evaluation plots...")
        # Get feature names from saved vectorizers for importance plot
        try:
            fold_models_dir = os.path.join(MODELS_DIR, f"fold_{fold}")
            word_vect = joblib.load(os.path.join(fold_models_dir, "word_vectorizer.joblib"))
            char_vect = joblib.load(os.path.join(fold_models_dir, "char_vectorizer.joblib"))
            feature_names = list(word_vect.get_feature_names_out()) + list(char_vect.get_feature_names_out())
        except Exception as e:
            print(f"  Warning: Could not load vectorizers for feature names: {e}")
            feature_names = [f"feature_{i}" for i in range(X_train.shape[1])]
            
        save_fold_plots(fold, y_test, y_pred_default, model.feature_importances_, feature_names)
        
        # 5. Persist model + metrics
        print(f"\n[5/5] Saving model and results...")
        save_fold_results(fold, model, metrics_default, metrics_tuned, best_threshold)
        
        fold_summaries.append({
            "fold"           : fold,
            "best_threshold" : best_threshold,
            "default"        : metrics_default,
            "tuned"          : metrics_tuned,
        })
        
    # ==========================================================================
    # GLOBAL OUT-OF-FOLD EVALUATION & TUNING
    # ==========================================================================
    oof_y_prob = np.concatenate(oof_probs)
    oof_y_true = np.concatenate(oof_labels)
    
    print("\n" + "=" * 65)
    print("  GLOBAL OUT-OF-FOLD EVALUATION (Default Threshold = 0.5)")
    print("=" * 65)
    global_default_metrics = calculate_metrics(oof_y_true, (oof_y_prob >= 0.5).astype(int), oof_y_prob)
    print_evaluation_report(f"{MODEL_NAME} (Global OOF - Default)", global_default_metrics)
    
    print("\n" + "=" * 65)
    print(f"  GLOBAL ACADEMIC INTEGRITY TUNING (FPR <= {TARGET_FPR*100:.1f}%)")
    print("=" * 65)
    global_tuned_threshold, global_tuned_metrics = tune_decision_threshold(oof_y_true, oof_y_prob, target_fpr=TARGET_FPR)
    print_evaluation_report(f"{MODEL_NAME} (Global OOF - Tuned @ {global_tuned_threshold:.3f})", global_tuned_metrics)
    
    # Generate and save plots (confusion matrix & ROC curve)
    from evaluate import plot_global_evaluation
    plot_global_evaluation(oof_y_true, oof_y_prob, global_tuned_threshold, MODEL_NAME, "rf")
    
    # ==========================================================================
    # CROSS-FOLD SUMMARY
    # ==========================================================================
    print("\n" + "=" * 65)
    print(f"  CROSS-FOLD SUMMARY - {MODEL_NAME}")
    print("=" * 65)
    
    metrics_keys = ["accuracy", "f1_score", "auc_roc", "precision",
                    "recall", "specificity", "false_positive_rate"]
    
    print(f"\n{'Metric':<30} {'Mean (default)':>15} {'Mean (tuned)':>14}")
    print("-" * 62)
    for key in metrics_keys:
        default_vals = [s["default"][key] for s in fold_summaries if s["default"][key] is not None]
        tuned_vals   = [s["tuned"][key]   for s in fold_summaries if s["tuned"][key]   is not None]
        mean_def = np.mean(default_vals) if default_vals else float("nan")
        mean_tun = np.mean(tuned_vals)   if tuned_vals   else float("nan")
        label = key.replace("_", " ").title()
        print(f"  {label:<28} {mean_def:>13.4f}  {mean_tun:>13.4f}")
        
    # Save cross-fold summary
    summary_path = os.path.join(OUTPUT_DIR, "rf_cross_fold_summary.json")
    summary_export = []
    for s in fold_summaries:
        summary_export.append({
            "fold"           : s["fold"],
            "best_threshold" : s["best_threshold"],
            "metrics_default": {k: s["default"][k] for k in metrics_keys},
            "metrics_tuned"  : {k: s["tuned"][k]   for k in metrics_keys},
        })
        
    global_results = {
        "model_name": MODEL_NAME,
        "global_default": {k: global_default_metrics[k] for k in metrics_keys},
        "global_tuned": {k: global_tuned_metrics[k] for k in metrics_keys},
        "global_tuned_threshold": global_tuned_threshold,
        "fold_results": summary_export
    }
    
    with open(summary_path, "w") as f:
        json.dump(global_results, f, indent=4)
    print(f"\nCross-fold summary saved -> {summary_path}")
    print("=" * 65)
    print("  Training complete!")
    print("=" * 65)

run_training()


## Step 5: Comparative Evaluation & Synthesis
Loads OOF evaluations across all 4 classifiers, displays a comparison table, and generates comparison visualisations in the workspace.

In [ ]:
def load_results():
    """Load cross-fold summaries from JSON files."""
    models_data = {}
    
    file_mapping = {
        "Logistic Regression": "logistic_cross_fold_summary.json",
        "Support Vector Machine": "svm_cross_fold_summary.json",
        "Multinomial Naive Bayes": "mnb_cross_fold_summary.json",
        "Random Forest": "rf_cross_fold_summary.json"
    }
    
    for name, filename in file_mapping.items():
        path = os.path.join(RESULTS_DIR, filename)
        if os.path.exists(path):
            with open(path, "r") as f:
                models_data[name] = json.load(f)
        else:
            print(f"Warning: Summary file {filename} not found at {path}.")
            
    return models_data

def compile_comparison_dataframe(models_data):
    """Transform the loaded JSON results into a clean pandas DataFrame for comparisons."""
    records = []
    metrics_keys = ["accuracy", "f1_score", "auc_roc", "precision", "recall", "specificity", "false_positive_rate"]
    
    for model_name, data in models_data.items():
        # Robust schema parsing (list of folds vs dict wrapper)
        if isinstance(data, list):
            fold_results = data
            thresholds = [f["best_threshold"] for f in fold_results if "best_threshold" in f]
            avg_threshold = np.mean(thresholds) if thresholds else 0.5
        elif isinstance(data, dict):
            fold_results = data["fold_results"]
            avg_threshold = data.get("global_tuned_threshold", 0.5)
        else:
            print(f"Error: Format of {model_name} results is unrecognized.")
            continue
            
        # Calculate mean metrics across all folds
        def_metrics = {}
        for key in metrics_keys:
            vals = [f["metrics_default"][key] for f in fold_results if key in f["metrics_default"]]
            def_metrics[key] = np.mean(vals) if vals else 0.0
            
        tuned_metrics = {}
        for key in metrics_keys:
            vals = [f["metrics_tuned"][key] for f in fold_results if key in f["metrics_tuned"]]
            tuned_metrics[key] = np.mean(vals) if vals else 0.0
            
        # 1. Default Threshold (0.5)
        rec_default = {
            "Model": model_name,
            "Setting": "Default (0.50)",
            "Accuracy": def_metrics["accuracy"] * 100,
            "F1-Score": def_metrics["f1_score"],
            "ROC-AUC": def_metrics["auc_roc"],
            "Precision": def_metrics["precision"] * 100,
            "Recall (Detection)": def_metrics["recall"] * 100,
            "Specificity": def_metrics["specificity"] * 100,
            "FPR": def_metrics["false_positive_rate"] * 100
        }
        records.append(rec_default)
        
        # 2. Tuned Safety Threshold
        rec_tuned = {
            "Model": model_name,
            "Setting": f"Tuned ({avg_threshold:.3f})",
            "Accuracy": tuned_metrics["accuracy"] * 100,
            "F1-Score": tuned_metrics["f1_score"],
            "ROC-AUC": tuned_metrics["auc_roc"],
            "Precision": tuned_metrics["precision"] * 100,
            "Recall (Detection)": tuned_metrics["recall"] * 100,
            "Specificity": tuned_metrics["specificity"] * 100,
            "FPR": tuned_metrics["false_positive_rate"] * 100
        }
        records.append(rec_tuned)
        
    return pd.DataFrame(records)

def plot_roc_auc_comparison(df_comparison):
    """Generate and save a bar chart comparing the ROC-AUC of all models."""
    plt.figure(figsize=(8, 5))
    
    # Filter default settings (AUC is the same for default/tuned, so we just pick one)
    df_auc = df_comparison[df_comparison["Setting"].str.contains("Default")].copy()
    df_auc = df_auc.sort_values(by="ROC-AUC", ascending=False)
    
    colors = sns.color_palette("viridis", len(df_auc))
    bars = plt.bar(df_auc["Model"], df_auc["ROC-AUC"], color=colors, edgecolor='grey', width=0.5)
    
    # Highlight value labels on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
        
    plt.ylim(0.90, 1.01)  # Focus on the high-performance range
    plt.ylabel("ROC-AUC Score")
    plt.title("Classifier Discrimination Capability (ROC-AUC)")
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGE_DIR, "model_comparison_roc_auc.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved ROC-AUC comparison plot -> {plot_path}")

def plot_safety_recall_comparison(df_comparison):
    """Generate and save a chart comparing the Recall (AI detection) under tuned safety threshold."""
    plt.figure(figsize=(9, 5.5))
    
    # Filter tuned safety settings
    df_tuned = df_comparison[df_comparison["Setting"].str.contains("Tuned")].copy()
    df_tuned = df_tuned.sort_values(by="Recall (Detection)", ascending=False)
    
    colors = sns.color_palette("magma", len(df_tuned))
    bars = plt.bar(df_tuned["Model"], df_tuned["Recall (Detection)"], color=colors, edgecolor='grey', width=0.5)
    
    # Add values on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
        
    plt.ylim(0, 110)
    plt.ylabel("Recall / AI Detection Rate (%)")
    plt.xlabel("Model")
    plt.title("Recall / Detection Rate Under Strict Academic Integrity safety (FPR <= 0.1%)")
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGE_DIR, "model_comparison_tuned_recall.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved Tuned Recall comparison plot -> {plot_path}")

def plot_fpr_default_vs_tuned(df_comparison):
    """Plot comparative FPR rates before and after safety threshold tuning."""
    plt.figure(figsize=(9, 5.5))
    
    # Restructure dataframe for grouped bar plotting
    df_plot = df_comparison.copy()
    df_plot["Setting Type"] = df_plot["Setting"].apply(lambda x: "Default (0.50)" if "Default" in x else "Tuned (FPR <= 0.1%)")
    
    ax = sns.barplot(
        data=df_plot,
        x="Model",
        y="FPR",
        hue="Setting Type",
        palette="muted"
    )
    
    # Add value annotations on top of the bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.3f}%",
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='center',
                        xytext=(0, 7),
                        textcoords='offset points',
                        fontweight='bold', fontsize=9)
            
    # Add red line for target FPR boundary
    plt.axhline(0.1, color='red', linestyle='--', linewidth=1.5, label="Safety Threshold Ceiling (0.1%)")
    
    plt.ylabel("False Positive Rate (%)")
    plt.title("False Positive Rate (False Accusations): Default vs Tuned Threshold")
    plt.legend(loc="upper right")
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGE_DIR, "model_comparison_fpr_reduction.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved FPR Comparison plot -> {plot_path}")

def main():
    print("=== Loading Model Summaries ===")
    models_data = load_results()
    
    if not models_data:
        print("No model results available. Please run the training scripts first.")
        return
        
    df_comparison = compile_comparison_dataframe(models_data)
    
    print("\n=== GLOBAL MODEL COMPARISON TABLE (CROSS-FOLD MEANS) ===")
    cols_to_show = ["Model", "Setting", "Accuracy", "F1-Score", "ROC-AUC", "Precision", "Recall (Detection)", "FPR"]
    print(df_comparison[cols_to_show].to_string(index=False))
    
    print("\n=== Generating Comparative Visualization Plots ===")
    plot_roc_auc_comparison(df_comparison)
    plot_safety_recall_comparison(df_comparison)
    plot_fpr_default_vs_tuned(df_comparison)
    print("\nVisualizations saved successfully to the results/ folder.")

models_data = load_results()
if models_data:
    comparison_df = compile_comparison_dataframe(models_data)
    print_comparison_markdown(comparison_df)
    save_comparison_plots(models_data)
